# M10 · Learning with sparse & implicit labels

_AFP-AI · Domain 1 · Ranking & Recommenders_

**Train from clicks and views without pretending missing means dislike.**

We build an in-batch softmax, apply popularity correction, and compare BPR margins. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
# Setup - CPU-only and deterministic.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(10)

## First, look at implicit feedback

Observed clicks are positives. Unobserved pairs may be unexposed, ignored, or delayed. Pairwise training uses $\log\sigma(s(u,i^+)-s(u,i^-))$.

In [ ]:
items = ["positive", "popular_neg", "rare_neg", "hard_neg"]
scores = np.array([3.0, 2.0, 1.0, 2.6])
q = np.array([0.10, 0.50, 0.05, 0.20])
df = pd.DataFrame({"item": items, "score": scores, "sample_q": q})

print(df)

## The in-batch softmax

For one positive, the raw loss is

$$\ell=-\log\frac{\exp(s^+)}{\sum_j \exp(s_j)}$$

Sampling correction subtracts $\log q(j)$ from each score.

### Step 1 - Compute raw positive probability

Every item in the batch competes for probability mass.

In [ ]:
exp_scores = np.exp(scores)
raw_prob = exp_scores[0] / exp_scores.sum()
raw_loss = -np.log(raw_prob)

print("raw positive probability:", round(raw_prob, 4))
print("raw loss:", round(raw_loss, 4))

assert raw_prob > 0.4

### Step 2 - Apply popularity correction

Subtracting $\log q$ reduces the advantage of frequently sampled negatives.

In [ ]:
corrected_scores = scores - np.log(q)
exp_corrected = np.exp(corrected_scores)
corr_prob = exp_corrected[0] / exp_corrected.sum()
corr_loss = -np.log(corr_prob)

print(pd.DataFrame({"item": items, "corrected_score": corrected_scores}).round(3))
print("corrected probability:", round(corr_prob, 4))

assert corr_prob != raw_prob

### Step 3 - Compute BPR loss against a hard negative

A hard negative has a high score, so the margin is small and the loss is larger.

In [ ]:
margin = scores[0] - scores[3]
bpr_prob = 1.0 / (1.0 + np.exp(-margin))
bpr_loss = -np.log(bpr_prob)

print("margin:", round(margin, 3))
print("BPR loss:", round(bpr_loss, 3))

assert bpr_loss > 0.5

## Visualize raw vs corrected logits

The correction changes which negatives are most competitive, because the sampler is part of the training distribution.

In [ ]:
x = np.arange(len(items))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(x - 0.18, scores, width=0.36, label="raw")
ax.bar(x + 0.18, corrected_scores, width=0.36, label="corrected")
ax.set_xticks(x)
ax.set_xticklabels(items, rotation=20)
ax.set_ylabel("logit")
ax.set_title("sampling correction")
ax.legend()
plt.show()

## Practice

Try each in the empty cell below it.

1. Change `sample_q` for `popular_neg` from 0.50 to 0.80 and recompute.
2. Add another hard negative with score 2.9 and observe BPR loss.
3. Compute IPS weights for propensities 0.9, 0.5, and 0.2.

In [ ]:
# Your turn:
